In [15]:
import optuna
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

In [16]:
train_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/feature_engineered_train.csv")
test_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/feature_engineered_test.csv")
rul_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/clean_rul.csv")

In [17]:
SEQ_LEN = 30  # typical: 20–50

def create_sequences(df, seq_len, feature_cols):
    sequences = []
    targets = []

    for unit_id in df["engine_id"].unique():
        engine_data = df[df["engine_id"] == unit_id]

        X_engine = engine_data[feature_cols].values
        y_engine = engine_data["rul"].values

        for i in range(len(X_engine) - seq_len):
            sequences.append(X_engine[i:i+seq_len])
            targets.append(y_engine[i+seq_len])

    return np.array(sequences), np.array(targets)

In [18]:
def create_sequences_last_only(df, seq_len, feature_cols):
    sequences = []
    targets = []

    for unit_id in df["engine_id"].unique():
        engine_data = df[df["engine_id"] == unit_id]

        X_engine = engine_data[feature_cols].values
        y_engine = engine_data["rul"].values

        if len(X_engine) >= seq_len:
            seq = X_engine[-seq_len:]
        else:
            padding = np.repeat(X_engine[0:1], seq_len - len(X_engine), axis=0)
            seq = np.vstack((padding, X_engine))

        sequences.append(seq)
        targets.append(y_engine[-1])  # 🔥 ONLY LAST RUL

    return np.array(sequences), np.array(targets)

In [19]:
feature_cols = [col for col in train_df.columns 
                if col not in ["engine_id", "cycle", "rul"]]

In [20]:
import torch
import torch.nn as nn

# Define model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # last timestep
        out = self.fc(out)
        return out.squeeze()

In [21]:
# Initialize device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
# Evaluation function
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_lstm(model, X, y):
    model.eval()

    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        preds = model(X_tensor).cpu().numpy()

    rmse = np.sqrt(mean_squared_error(y, preds))
    mae = mean_absolute_error(y, preds)
    r2 = r2_score(y, preds)
    
    return rmse, mae, r2

In [23]:
# ====================
# Test data
# ====================
import numpy as np

def create_test_sequences(df, seq_len, feature_cols):
    sequences = []

    for unit_id in df["engine_id"].unique():
        engine_data = df[df["engine_id"] == unit_id]

        X_engine = engine_data[feature_cols].values

        if len(X_engine) >= seq_len:
            seq = X_engine[-seq_len:]
        else:
            # 🔥 PAD with zeros at the beginning
            padding = np.zeros((seq_len - len(X_engine), X_engine.shape[1]))
            seq = np.vstack((padding, X_engine))

        sequences.append(seq)

    return np.array(sequences)

In [24]:
# Setup MLFlow
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("/Users/apple/Documents/Projects/Nasa RUL MLE/mlruns")
mlflow.set_experiment("NASA_Turbofan_RUL")

# Define optuna objective function
def objective(trial):

    # Hyperparameters to tune
    seq_len = trial.suggest_int("seq_len", 20, 50)
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])

    # Recreate sequences (IMPORTANT)
    X_seq, y_seq = create_sequences(train_df, seq_len, feature_cols)
    #X_seq, y_seq = create_sequences_last_only(train_df, seq_len, feature_cols)

    from sklearn.model_selection import train_test_split
    X_train, X_val, y_train, y_val = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

    # Convert to PyTorch
    train_dataset = torch.utils.data.TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    # Model
    model = LSTMModel(
        input_size=len(feature_cols),
        hidden_size=hidden_size,
        num_layers=num_layers
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # MLflow tracking
    with mlflow.start_run(nested=True):

        mlflow.log_params({
            "seq_len": seq_len,
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "learning_rate": lr,
            "batch_size": batch_size
        })

        # Training loop (short!)
        EPOCHS = 8
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0

            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)

                optimizer.zero_grad()
                preds = model(xb)
                loss = criterion(preds, yb)

                loss.backward()
                optimizer.step()

                total_loss += loss.item()

        # Validation
        model.eval()
        with torch.no_grad():
            X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
            preds = model(X_val_tensor).cpu().numpy()

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        mae = mean_absolute_error(y_val, preds)
        r2 = r2_score(y_val, preds)

        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [25]:
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="lstm_optuna"):
    study.optimize(objective, n_trials=20)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_rmse", study.best_value)

[I 2026-04-09 15:29:53,307] A new study created in memory with name: no-name-86154938-f473-4316-88e2-5a78799e2272
[I 2026-04-09 15:30:30,386] Trial 0 finished with value: 11.594893329569311 and parameters: {'seq_len': 42, 'hidden_size': 128, 'num_layers': 1, 'learning_rate': 0.000655726471190454, 'batch_size': 32}. Best is trial 0 with value: 11.594893329569311.
[I 2026-04-09 15:30:39,808] Trial 1 finished with value: 14.195546239388538 and parameters: {'seq_len': 43, 'hidden_size': 32, 'num_layers': 1, 'learning_rate': 0.0077692991050292, 'batch_size': 64}. Best is trial 0 with value: 11.594893329569311.
[I 2026-04-09 15:30:48,170] Trial 2 finished with value: 12.330269691571944 and parameters: {'seq_len': 20, 'hidden_size': 32, 'num_layers': 1, 'learning_rate': 0.00429361886491741, 'batch_size': 32}. Best is trial 0 with value: 11.594893329569311.
[I 2026-04-09 15:31:26,371] Trial 3 finished with value: 9.203067968832467 and parameters: {'seq_len': 46, 'hidden_size': 128, 'num_layers

In [26]:
print("Best params:", study.best_params)
print("Best RMSE:", study.best_value)

Best params: {'seq_len': 40, 'hidden_size': 64, 'num_layers': 3, 'learning_rate': 0.0035556213925994093, 'batch_size': 32}
Best RMSE: 3.712672169262649


In [27]:
# ==============================================
# Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_params

# recreate sequences
X_seq, y_seq = create_sequences(train_df, best_params["seq_len"], feature_cols)
#X_seq, y_seq = create_sequences_last_only(train_df, best_params["seq_len"], feature_cols)

# split again
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# build model
best_model = LSTMModel(
    input_size=len(feature_cols),
    hidden_size=best_params["hidden_size"],
    num_layers=best_params["num_layers"]
).to(device)

optimizer = torch.optim.Adam(best_model.parameters(), lr=best_params["learning_rate"])
criterion = torch.nn.MSELoss()

EPOCHS = 20  # longer than tuning!

best_val_rmse = float("inf")
patience = 3
counter = 0

best_model_state = None

for epoch in range(EPOCHS):
    best_model.train()
    total_loss = 0

    for i in range(0, len(X_train), best_params["batch_size"]):
        xb = torch.tensor(X_train[i:i+best_params["batch_size"]], dtype=torch.float32).to(device)
        yb = torch.tensor(y_train[i:i+best_params["batch_size"]], dtype=torch.float32).to(device)

        optimizer.zero_grad()
        preds = best_model(xb)
        loss = criterion(preds, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(best_model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    # Evaluate on validation set
    val_rmse, val_mae, val_r2 = evaluate_lstm(best_model, X_val, y_val)

    print(f"Epoch {epoch+1}, Train Loss: {total_loss:.2f}, Val RMSE: {val_rmse:.4f}")

    # Early stopping logic
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_model_state = best_model.state_dict()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("⛔ Early stopping triggered")
        break

best_model.load_state_dict(best_model_state)

Epoch 1, Train Loss: 1274274.37, Val RMSE: 25.3993
Epoch 2, Train Loss: 149791.62, Val RMSE: 13.7430
Epoch 3, Train Loss: 55796.10, Val RMSE: 8.4609
Epoch 4, Train Loss: 28353.57, Val RMSE: 6.3509
Epoch 5, Train Loss: 15675.20, Val RMSE: 5.4207
Epoch 6, Train Loss: 9763.20, Val RMSE: 4.4765
Epoch 7, Train Loss: 6248.04, Val RMSE: 4.3192
Epoch 8, Train Loss: 4697.26, Val RMSE: 3.2372
Epoch 9, Train Loss: 3506.81, Val RMSE: 2.9242
Epoch 10, Train Loss: 2728.61, Val RMSE: 2.7477
Epoch 11, Train Loss: 2407.84, Val RMSE: 2.4983
Epoch 12, Train Loss: 2189.88, Val RMSE: 2.4342
Epoch 13, Train Loss: 1743.15, Val RMSE: 2.1690
Epoch 14, Train Loss: 1637.27, Val RMSE: 2.3137
Epoch 15, Train Loss: 1413.49, Val RMSE: 1.9257
Epoch 16, Train Loss: 1189.46, Val RMSE: 1.9864
Epoch 17, Train Loss: 1161.59, Val RMSE: 1.9307
Epoch 18, Train Loss: 932.96, Val RMSE: 1.6975
Epoch 19, Train Loss: 939.35, Val RMSE: 1.5123
Epoch 20, Train Loss: 831.35, Val RMSE: 1.7668


<All keys matched successfully>

In [28]:
X_test_seq = create_test_sequences(test_df, best_params["seq_len"], feature_cols)
y_test = rul_df["rul"].values

rmse_test, mae_test, r2_test = evaluate_lstm(best_model, X_test_seq, y_test)

print("Final LSTM performance on test data:")
print("RMSE:", rmse_test)
print("MAE:", mae_test)
print("R²:", r2_test)

# Log final model
with mlflow.start_run(run_name="best_lstm_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse_test, "mae": mae_test, "r2": r2_test})
    mlflow.pytorch.log_model(best_model, name="model")

2026/04/09 15:46:11 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Final LSTM performance on test data:
RMSE: 17.45867312638576
MAE: 12.834778785705566
R²: 0.8234925866127014
